In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder.appName("EmployeeRDDAnalysis").getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/15 13:13:58 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/15 13:14:15 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [3]:
sc = spark.sparkContext

In [4]:
rdd = sc.textFile("employees.csv")

In [5]:
 header = rdd.first()
rdd = rdd.filter(lambda x: x != header)

In [7]:
def convert(row):
    x = row.split(",")
    return (int(x[0]), x[1], x[2], int(x[3]))

employees = rdd.map(convert)

In [14]:
sorted_employees = employees.sortBy(lambda x: x[3], ascending=False)

for emp in sorted_employees.collect():
    print(emp)
sorted_employees.saveAsTextFile("output/descending_salary")

(4, 'Priya', 'Finance', 70000)
(3, 'Neha', 'IT', 65000)
(7, 'Rohit', 'Finance', 60000)
(1, 'Amit', 'IT', 55000)
(5, 'Karan', 'IT', 50000)
(6, 'Simran', 'HR', 45000)
(2, 'Rahul', 'HR', 40000)


In [15]:
dept_salary = employees.map(lambda x: (x[2], x[3])) \
                       .reduceByKey(lambda a, b: a + b)

for dept in dept_salary.collect():
    print(dept)
dept_salary.saveAsTextFile("output/dept_salary")

('IT', 170000)
('HR', 85000)
('Finance', 130000)


In [11]:
top3 = sorted_employees.take(3)
for emp in top3:
    print(emp)


(4, 'Priya', 'Finance', 70000)
(3, 'Neha', 'IT', 65000)
(7, 'Rohit', 'Finance', 60000)


In [12]:
sc.parallelize(
    [f"{emp[0]},{emp[1]},{emp[2]},{emp[3]}" for emp in top3]
).saveAsTextFile("output/top3_employees")